In [1]:
import os
import glob
import json
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, f1_score, classification_report

# Set pandas display options 
pd.set_option('display.max_rows', 20)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 300)

In [2]:
def clean_and_sort_imports(import_list):
    # Split each import statement into lines
    imports = import_list.split('\n')
    
    # Remove empty lines, strip whitespace, and remove duplicates
    unique_imports = list(dict.fromkeys(line.strip() for line in imports if line.strip()))
    
    # Sort imports by length, then alphabetically for imports of the same length
    sorted_imports = sorted(unique_imports, key=lambda x: (len(x), x))
    
    return sorted_imports

# Example usage:
imports = """
import pandas as pd
import pandas as pd
import pandas as pd
import pandas as pd
import json
import os
import glob
import json
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, f1_score, classification_report
import matplotlib.pyplot as plt
"""

cleaned_sorted_imports = clean_and_sort_imports(imports)

# Print the cleaned and sorted imports
for imp in cleaned_sorted_imports:
    print(imp)

import os
import glob
import json
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, f1_score, classification_report


In [3]:
# Load the TSV file
tsv_file = 'dataset/mine/test.tsv'
# df_tsv = pd.read_csv(tsv_file, sep='\t', names=['query-id', 'corpus-id', 'score'])
df_tsv = pd.read_csv(tsv_file, sep='\t', header=0)

In [4]:
# Display info about the DataFrame
print("\nDataFrame Info:")
print(df_tsv.info())

# Display basic statistics of the DataFrame
print("\nDataFrame Description:")
print(df_tsv.describe())

# Display value counts for each column
for column in df_tsv.columns:
    print(f"\nValue counts for {column}:")
    print(df_tsv[column].value_counts().head())

# # Optional: Save to CSV for easy viewing in spreadsheet software
# df_tsv.to_csv('test_tsv_data.csv', index=False)
# print("\nDataFrame saved to 'test_tsv_data.csv'")# Display the first few rows of the DataFrame

# Display the first few rows of the DataFrame
print("First few rows of the DataFrame:")
# df_tsv.head()
df_tsv


DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3870 entries, 0 to 3869
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   query-id   3870 non-null   object
 1   corpus-id  3870 non-null   object
 2   score      3870 non-null   int64 
dtypes: int64(1), object(2)
memory usage: 90.8+ KB
None

DataFrame Description:
             score
count  3870.000000
mean      0.394574
std       0.675703
min       0.000000
25%       0.000000
50%       0.000000
75%       1.000000
max       2.000000

Value counts for query-id:
query-id
sigir-20147     153
sigir-20141     143
sigir-201518    118
sigir-20153     118
sigir-20146     108
Name: count, dtype: int64

Value counts for corpus-id:
corpus-id
NCT02134652    5
NCT00455468    5
NCT02269761    4
NCT01512277    4
NCT00264901    4
Name: count, dtype: int64

Value counts for score:
score
0    2764
1     685
2     421
Name: count, dtype: int64
First few rows of the DataFrame

,query-id,corpus-id,score
0,sigir-20141,NCT00000408,0
1,sigir-20141,NCT00000492,1
2,sigir-20141,NCT00000501,0
3,sigir-20141,NCT00001853,0
4,sigir-20141,NCT00004727,0
...,...,...,...
3865,sigir-20159,NCT02459171,0
3866,sigir-20159,NCT02459327,0
3867,sigir-20159,NCT02498964,0
3868,sigir-20159,NCT02612896,0


In [5]:
# Read the JSONL file from the dataset
# This file is part of the TrialGPT dataset and contains query information
# The 'queries.jsonl' file likely includes patient information
jsonl_file = 'dataset/mine/queries.jsonl'
queries = []

# Open the file and read it line by line
with open(jsonl_file, 'r') as file:
    for line in file:
        # Parse each line as a JSON object and append it to the queries list
        # This approach is memory-efficient for large files as it processes one line at a time
        queries.append(json.loads(line))

# Convert the list of dictionaries (parsed JSON objects) to a pandas DataFrame
# This transformation allows for easier data manipulation and analysis using pandas functions
df_queries = pd.DataFrame(queries)

# Display the entire DataFrame
# This allows us to inspect the structure and content of the queries data from the dataset
df_queries

,_id,text
0,sigir-20141,"A 58-year-old African-American woman presents to the ER with episodic pressing/burning anterior chest pain that began two days earlier for the first time in her life. The pain started while she was walking, radiates to the back, and is accompanied by nausea, diaphoresis and mild dyspnea, but is ..."
1,sigir-20142,"An 8-year-old male presents in March to the ER with fever up to 39 C, dyspnea and cough for 2 days. He has just returned from a 5 day vacation in Colorado. Parents report that prior to the onset of fever and cough, he had loose stools. He denies upper respiratory tract symptoms. On examination h..."
2,sigir-20143,"A 58-year-old nonsmoker white female with mild exertional dyspnea and occasional cough is found to have a left lung mass on chest x-ray. She is otherwise asymptomatic. A neurologic examination is unremarkable, but a CT scan of the head shows a solitary mass in the right frontal lobe."
3,sigir-20144,"A 2-year-old boy is brought to the emergency department by his parents for 5 days of high fever and irritability. The physical exam reveals conjunctivitis, strawberry tongue, inflammation of the hands and feet, desquamation of the skin of the fingers and toes, and cervical lymphadenopathy with t..."
4,sigir-20145,A 56-year-old female on 20th day post-left mastectomy presents to the emergency department complaining of shortness of breath and malaise. The patient says that she has remained in bed for the last two weeks. The physical examination reveals tenderness on the left upper thoracic wall and right c...
...,...,...
55,sigir-201526,"A 28 yo female G1P0A0 is admitted to the Ob/Gyn service for non-ruptured ectopic pregnancy. Past medical history is remarkable for obesity, a non-complicated appendectomy at age 8, infertility treatment for the past 3 years, and pelvic laparoscopy during which minor right Fallopian tube adhesion..."
56,sigir-201527,"A 15 yo girl accompanied by her mother is referred for evaluation by the school. The girl has more than expected absences in the last three month, appears to be constantly tired and sleepy in class. Her mother assures the girl is well fed, and getting the proper sleep at night but admits the gir..."
57,sigir-201528,"A previously healthy 8-year-old boy presents with a complaint of right lower extremity pain and fever. He reports limping for the past two days. The parents report no previous trauma, but do remember a tick bite during a summer visit to Maryland several months ago. They do not remember observing..."
58,sigir-201529,"A 4-year-old girl presents with persistent fever for the past week. The parents report a spike at 104°F. The parents brought the child to the emergency room when they noticed erythematous rash on the girl's trunk. Physical examination reveals strawberry red tongue, red and cracked lips, and swol..."


In [6]:
# Read the corpus.jsonl file
# corpus.jsonl contains detailed information about clinical trials
# Each line in this file is a JSON object representing a single trial
# The structure includes fields like '_id' (trial ID), 'title', 'text', and 'metadata'
corpus_file = 'dataset/mine/corpus.jsonl'

# Use pandas to read the JSONL file
# The 'lines=True' parameter tells pandas to read the file as JSON Lines format
# where each line is a separate JSON object
df_corpus = pd.read_json(corpus_file, lines=True)

# Display the entire DataFrame
df_corpus

# Note on corpus.jsonl structure:
# Each entry in corpus.jsonl looks like this:
# {
# "_id": nct_id,
# "title": title,
# "metadata": {
#     "phase": phase,
#     "drugs": str(drugs_list),
#     "drugs_list": drugs_list,
#     "diseases": str(diseases_list),
#     "diseases_list": diseases_list,
#     "enrollment": enrollment_value,
#     "inclusion_criteria": inclusion_text,
#     "exclusion_criteria": exclusion_text,
#     "brief_summary": brief_summary_text,
#     "detailed_description": detailed_description_text
# }
# }

,_id,title,metadata
0,NCT00000102,Congenital Adrenal Hyperplasia: Calcium Channels as Therapeutic Targets,"{'phase': 'Phase 1/Phase 2', 'drugs': '['Nifedipine']', 'drugs_list': ['Nifedipine'], 'diseases_list': ['Congenital Adrenal Hyperplasia'], 'enrollment': '0', 'inclusion_criteria': '- diagnosed with Congenital Adrenal Hyperplasia (CAH) - normal ECG during baseline evaluation', 'exclusion_criteria..."
1,NCT00000104,Does Lead Burden Alter Neuropsychological Development?,"{'phase': 'N/A', 'drugs': '['ERP measures of attention and memory']', 'drugs_list': ['ERP measures of attention and memory'], 'diseases_list': ['Lead Poisoning'], 'enrollment': '0', 'inclusion_criteria': '- Pregnant mothers of the Phillips neighborhood in Minneapolis, Minnesota. Subject recruitm..."
2,NCT00000105,Vaccination With Tetanus and KLH to Assess Immune Responses.,"{'phase': 'N/A', 'drugs': '['Intracel KLH Vaccine', 'Biosyn KLH', 'Montanide ISA51', 'Tetanus toxoid']', 'drugs_list': ['Intracel KLH Vaccine', 'Biosyn KLH', 'Montanide ISA51', 'Tetanus toxoid'], 'diseases_list': ['Cancer'], 'enrollment': '112', 'inclusion_criteria': '- Patients must have a diag..."
3,NCT00000106,41.8 Degree Centigrade Whole Body Hyperthermia for the Treatment of Rheumatoid Diseases,"{'phase': 'N/A', 'drugs': '['Whole body hyperthermia unit']', 'drugs_list': ['Whole body hyperthermia unit'], 'diseases_list': ['Rheumatic Diseases'], 'enrollment': '0', 'inclusion_criteria': '- Patients are required to meet the criteria of the American College of Rheumatology (ACR)for rheumatoi..."
4,NCT00000107,Body Water Content in Cyanotic Congenital Heart Disease,"{'phase': 'N/A', 'drugs': '[]', 'drugs_list': [], 'diseases_list': ['Heart Defects, Congenital'], 'enrollment': '0', 'inclusion_criteria': '- Resting blood pressure below 140/90', 'exclusion_criteria': '', 'brief_summary': 'Adults with cyanotic congenital heart disease have elevated levels of pl..."
...,...,...,...
204850,NCT02634177,Genecept Assay™ vs. Treatment-as-Usual to Evaluate Efficacy of Assay-Guided Treatment in Adults With Major Depressive Disorder,"{'phase': 'N/A', 'drugs': '['Assay-guided treatment (AGT)', 'Treatment-as-usual (TAU)']', 'drugs_list': ['Assay-guided treatment (AGT)', 'Treatment-as-usual (TAU)'], 'diseases_list': ['Major Depressive Disorder'], 'enrollment': '300', 'inclusion_criteria': '1. Age 18-75 years 2. Ability to under..."
204851,NCT02634190,Clinical Evaluation of the APTIMA® HPV Assay and Comparison With the HR HC2® Test Using LBC ThinPrep® Specimens,"{'phase': 'N/A', 'drugs': '['Thinprep® LBC', 'APTIMA® HPV Assay', 'HR HC2® HPV DNA', 'Colposcopy']', 'drugs_list': ['Thinprep® LBC', 'APTIMA® HPV Assay', 'HR HC2® HPV DNA', 'Colposcopy'], 'diseases_list': ['Human Papilloma Virus Infection'], 'enrollment': '10000', 'inclusion_criteria': '', 'excl..."
204852,NCT02634203,Riociguat Versus Balloon Pulmonary Angioplasty in Non-operable Chronic thromboEmbolic Pulmonary Hypertension,"{'phase': 'N/A', 'drugs': '['Balloon Pulmonary Angioplasty (BPA)', 'Riociguat']', 'drugs_list': ['Balloon Pulmonary Angioplasty (BPA)', 'Riociguat'], 'diseases_list': ['Chronic Thromboembolic Pulmonary Hypertension'], 'enrollment': '124', 'inclusion_criteria': '- 18 to 80 years of age at Visit 1..."
204853,NCT02634216,Effects of Capros in Patients With Type-1 Diabetes,"{'phase': 'N/A', 'drugs': '['Capros']', 'drugs_list': ['Capros'], 'diseases_list': ['Type I Diabetes'], 'enrollment': '20', 'inclusion_criteria': '- Subjects must be 10 - 40 yrs. of age - Type 1 Diabetes using Continuous Glucose Monitoring (CGM) for at least the last 3 months - Less than 10% var..."


In [7]:
import pandas as pd
import json

# Read the JSON file
with open('results/retrieved_trials.json', 'r') as file:
    data = json.load(file)

# Create a list to store the flattened data
flattened_data = []

# Flatten the nested structure
for item in data:
    patient_id = item['patient_id']
    patient = item['patient']
    
    for key in ['0', '1', '2']:
        for trial in item[key]:
            trial_data = {
                'patient_id': patient_id,
                'patient': patient,
                'section': key,
                **trial
            }
            flattened_data.append(trial_data)

# Create a DataFrame
df_retrival = pd.DataFrame(flattened_data)

In [8]:
# Display the first few rows and basic information about the DataFrame
# print(df_retrival.head())
# print(df_retrival.info())
df_retrival

,patient_id,patient,section,nct_id,brief_title,phase,drugs,drugs_list,diseases,diseases_list,enrollment,inclusion_criteria,exclusion_criteria,brief_summary,NCTID,total_score,bm25_score,medcpt_score
0,sigir-20141,"<0.> A 58-year-old African-American woman presents to the ER with episodic pressing/burning anterior chest pain that began two days earlier for the first time in her life. <1.> The pain started while she was walking, radiates to the back, and is accompanied by nausea, diaphoresis and mild dyspne...",0,NCT00208845,Hypernet- Hypertension Screening,N/A,[],[],,[Asymptomatic Hypertension],100,- asymptomatic hypertension,- cardiac symptoms,"Identify untreated, asymptomatic hypertension in ER patients",NCT00208845,0.091726,0.041667,0.050059
1,sigir-20141,"<0.> A 58-year-old African-American woman presents to the ER with episodic pressing/burning anterior chest pain that began two days earlier for the first time in her life. <1.> The pain started while she was walking, radiates to the back, and is accompanied by nausea, diaphoresis and mild dyspne...",0,NCT00173004,Angiotensinogen Gene and Hypertension,N/A,[],[],,[Hypertension],0,- Patients with hypertension,- Patients with hypertension and reluctant for the study,"Genetic studies of hypertension, focusing on renin-angiotensin system genes and other interacting genes",NCT00173004,0.077381,0.035714,0.041667
2,sigir-20141,"<0.> A 58-year-old African-American woman presents to the ER with episodic pressing/burning anterior chest pain that began two days earlier for the first time in her life. <1.> The pain started while she was walking, radiates to the back, and is accompanied by nausea, diaphoresis and mild dyspne...",0,NCT01333683,Tinnitus and Arterial Hypertension,N/A,[],[],,"[Tinnitus, Hearing Loss, Arterial Hypertension]",100,- age between 40 and 50 - arterial hypertension for group 1 - at least 5 years standing arterial hypertension,- chronic noise exposure - metabolic diseases - family antecedents of hearing loss (except for presbycusis) - pregnant women - use of ototoxic drugs (except for anti-hypertensives),"Many authors link tinnitus to arterial hypertension. The aim of this study is to establish a possible relationship between them, analyze the severity of tinnitus related to arterial hypertension and analyze a possible influence of ototoxic drugs used to treat arterial hypertension",NCT01333683,0.070346,0.047619,0.022727
3,sigir-20141,"<0.> A 58-year-old African-American woman presents to the ER with episodic pressing/burning anterior chest pain that began two days earlier for the first time in her life. <1.> The pain started while she was walking, radiates to the back, and is accompanied by nausea, diaphoresis and mild dyspne...",0,NCT00134849,Hypertension in Management of Military Medicine,N/A,[],[],,[Hypertension],1000,- Newly diagnosed hypertension,,This study is a retrospective database review of primary care appointments of patients' charts with new diagnoses of hypertension.,NCT00134849,0.060036,0.027778,0.032258
4,sigir-20141,"<0.> A 58-year-old African-American woman presents to the ER with episodic pressing/burning anterior chest pain that began two days earlier for the first time in her life. <1.> The pain started while she was walking, radiates to the back, and is accompanied by nausea, diaphoresis and mild dyspne...",0,NCT00005149,Genetic and Environmental Determinants of Hypertension,N/A,[],[],,"[Cardiovascular Diseases, Heart Diseases, Hypertension]",0,,,To determine the pathophysiology of different types of essential hypertension by identifying the discrete effects of major genes and environmental variables as determinants of the subtypes of essential hypertension.,NCT00005149,0.056268,0.019231,0.037037
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299995,sigir-201530,"<0.> A 47 year old male who fell on his outstretched left arm presents with pain and bruising on the inside and outside of the elbow, swelling, and inabili

In [9]:
import pandas as pd

# Perform the left join
merged_df = df_retrival.merge(df_tsv, 
                              left_on=['patient_id', 'nct_id'], 
                              right_on=['query-id', 'corpus-id'], 
                              how='left')

# Fill NaN values with 0
merged_df['score'] = merged_df['score'].fillna(0)

# Count total scores 1 and 2 for each query-id in df_tsv
tsv_counts = df_tsv.groupby('query-id')['score'].value_counts().unstack(fill_value=0)
tsv_counts = tsv_counts.rename(columns={1: 'total_score_1', 2: 'total_score_2'})

# Count scores 1 and 2 that made it to merged_df for each patient_id
merged_counts = merged_df.groupby('patient_id')['score'].value_counts().unstack(fill_value=0)
merged_counts = merged_counts.rename(columns={1.0: 'found_score_1', 2.0: 'found_score_2'})

# Combine the counts
result = pd.concat([tsv_counts, merged_counts], axis=1).fillna(0)

# Calculate missing counts and percentages
result['missing_score_1'] = result['total_score_1'] - result['found_score_1']
result['missing_score_2'] = result['total_score_2'] - result['found_score_2']
result['percent_missing_1'] = (result['missing_score_1'] / result['total_score_1'] * 100).fillna(0)
result['percent_missing_2'] = (result['missing_score_2'] / result['total_score_2'] * 100).fillna(0)

# After calculating the result DataFrame as before

# Format the columns
result = result.astype({
    'total_score_1': 'int',
    'found_score_1': 'int',
    'missing_score_1': 'int',
    'total_score_2': 'int',
    'found_score_2': 'int',
    'missing_score_2': 'int'
})

# Format percentage columns to 1 decimal place
result['percent_missing_1'] = result['percent_missing_1'].round(1)
result['percent_missing_2'] = result['percent_missing_2'].round(1)

# Display the formatted results
pd.set_option('display.float_format', '{:.1f}'.format)
# print(result)

# Reset display options
pd.reset_option('display.float_format')

# Overall statistics (formatted as integers)
total_1 = int(result['total_score_1'].sum())
total_2 = int(result['total_score_2'].sum())
missing_1 = int(result['missing_score_1'].sum())
missing_2 = int(result['missing_score_2'].sum())
percent_missing_1 = (missing_1 / total_1 * 100) if total_1 > 0 else 0
percent_missing_2 = (missing_2 / total_2 * 100) if total_2 > 0 else 0

print("\nOverall Statistics:")
print(f"Total Score 1: {total_1}")
print(f"Total Score 2: {total_2}")
print(f"Missing Score 1: {missing_1} ({percent_missing_1:.1f}%)")
print(f"Missing Score 2: {missing_2} ({percent_missing_2:.1f}%)")


Overall Statistics:
Total Score 1: 685
Total Score 2: 421
Missing Score 1: 181 (26.4%)
Missing Score 2: 82 (19.5%)


In [10]:
pd.set_option('display.max_rows', None)  # Show all rows

In [11]:
result

score,0,total_score_1,total_score_2,0.0,found_score_1,found_score_2,missing_score_1,missing_score_2,percent_missing_1,percent_missing_2
sigir-20141,102.0,27,14,4985,10,5,17,9,63.0,64.3
sigir-201410,44.0,12,4,4984,12,4,0,0,0.0,0.0
sigir-201411,34.0,6,2,4999,0,1,6,1,100.0,50.0
sigir-201412,43.0,4,6,4990,4,6,0,0,0.0,0.0
sigir-201413,46.0,14,3,4990,8,2,6,1,42.9,33.3
sigir-201414,62.0,5,6,4995,1,4,4,2,80.0,33.3
sigir-201415,6.0,4,3,4993,4,3,0,0,0.0,0.0
sigir-201416,51.0,6,1,4998,2,0,4,1,66.7,100.0
sigir-201417,37.0,11,2,4990,9,1,2,1,18.2,50.0
sigir-201418,59.0,13,2,4992,7,1,6,1,46.2,50.0


In [12]:
pd.reset_option('display.max_rows')

In [13]:
# merged_df

sorted_merged_df = merged_df[merged_df['score'] != 0].sort_values(
    by=['patient_id', 'score', 'nct_id'], 
    ascending=[True, False, True]
)

sorted_merged_df

,patient_id,patient,section,nct_id,brief_title,phase,drugs,drugs_list,diseases,diseases_list,enrollment,inclusion_criteria,exclusion_criteria,brief_summary,NCTID,total_score,bm25_score,medcpt_score,query-id,corpus-id,score
3142,sigir-20141,"<0.> A 58-year-old African-American woman presents to the ER with episodic pressing/burning anterior chest pain that began two days earlier for the first time in her life. <1.> The pain started while she was walking, radiates to the back, and is accompanied by nausea, diaphoresis and mild dyspne...",0,NCT00005127,Muscatine Heart Study,N/A,[],[],,"[Cardiovascular Diseases, Coronary Disease, Hypertension, Heart Diseases]",0,,,To conduct longitudinal and cross-sectional studies of risk factors for coronary heart disease and hypertension in school age children and adults who had been examined in previous screens.,NCT00005127,0.001647,0.000974,0.000673,sigir-20141,NCT00005127,2.0
45,sigir-20141,"<0.> A 58-year-old African-American woman presents to the ER with episodic pressing/burning anterior chest pain that began two days earlier for the first time in her life. <1.> The pain started while she was walking, radiates to the back, and is accompanied by nausea, diaphoresis and mild dyspne...",0,NCT00373828,Non-cardiac Chest Pain Evaluation and Treatment Study (CARPA) - Part 1: Diagnosis.,N/A,[],[],,"[Non-Cardiac Chest Pain, Undiagnosed Chest Pain, Musculoskeletal Chest Pain, Ischemic Heart Disease]",302,"- Acute episode of chest pain of less than 7 days duration as primary reason for admission to a chest pain clinic. - Admitted to a chest pain clinic, suspected of acute coronary infarction, but with a negative diagnosis confirmed by normal coronary enzymes and normal ECG. - Pain arising from the...","- Acute coronary syndrome. - Percutaneous Coronary Intervention. - Coronary Artery Bypass Grafting. - Other disease, diagnosed during this admission, which is likely to have caused the acute episode of chest pain. - No written consent. - Inflammatory joint disease. - Diabetes mellitus, type I. -...","The overall aim of the project is to evaluate diagnosis and treatment of chest pain originating from the musculoskeletal system. Specifically, we wish to investigate prevalence and character of such chest pain in a population of patients with acute chest pain, admitted to a university hospital b...",NCT00373828,0.029584,0.016667,0.012917,sigir-20141,NCT00373828,2.0
736,sigir-20141,"<0.> A 58-year-old African-American woman presents to the ER with episodic pressing/burning anterior chest pain that began two days earlier for the first time in her life. <1.> The pain started while she was walking, radiates to the back, and is accompanied by nausea, diaphoresis and mild dyspne...",0,NCT00952744,Investigation of the Biomarker Copeptin in Patients With Acute Myocardial Infarction,N/A,[],[],,[Acute Coronary Syndromes],2071,"- The subject must be 18 years of age or older. - The subject must present to the Emergency Department with symptoms consistent with acute coronary syndromes (e.g., chest discomfort/pain, squeezing/fullness in the chest, pain radiating to left or both arms, jaw pain, pain in the back/neck/stomac...","- The patient is unable to provide consent or understand the consent form. - The ACS symptoms are clearly not the result of ACS (i.e., penetrating wounds, crush injury, etc.)","While troponin is not detectable until several hours after an Acute Myocardial Infarction (AMI), copeptin is expected to be elevated very early after an AMI. A combination of both markers for the diagnosis of AMI early after the event is therefore expected to be advantageous.",NCT00952744,0.005990,0.001529,0.004461,sigir-20141,NCT00952744,2.0
3678,sigir-20141,"<0.> A 58-year-old African-American woman presents to the ER with episodic pressing/burning anterior chest pain that began two days earlier for the first time in her life. <1.> The pain started while she was walking, radiates to the back, and is accompanied by nausea